# Low-Light Image Enhancement — Kaggle workflow

This notebook orchestrates the scripts and YAML configurations in the GitHub repository. Each computational stage is opt-in; dataset acquisition is performed **manually from the original publishers** before mounting the data as Kaggle Inputs. The notebook does not silently download or redistribute image datasets.

1. Get the dataset archives from the [official URLs in the dataset section below](#dataset-downloads-and-exact-folder-layout).
2. Extract and arrange the raw data exactly as shown, without renaming paired image files.
3. Upload the resulting **`data/raw/` contents** as a private Kaggle Dataset (respecting publisher terms), or use a lawful pre-existing mounted input. Set `RAW_DATA_DIRECTORY` accordingly.
4. Configure the GitHub repository URL and enable `RUN_MANIFEST_GENERATION` and `RUN_OFFLINE_PREPROCESSING` to produce the CSV manifests and 256 × 256 PNGs.
5. Enable training, log parsing, or exports only when explicitly needed.

For all four dataset sources, the repository `README.md` also documents the directory layout. Training may require compatible GPU acceleration and substantial execution time. The original experiment protocol is described in `docs/experiments.md`.


In [ ]:
# User settings — edit before executing. All computational stages are opt-in.
GITHUB_REPOSITORY_URL = "https://github.com/cosimo-curri/low-light-enhancement-cross-dataset.git"  # Available after repository publication
RUN_MANIFEST_GENERATION = False
RUN_OFFLINE_PREPROCESSING = False
RUN_TRAINING_EVALUATION = False
RUN_LOG_PARSING = False
RUN_QUALITATIVE_EXPORT = False
RUN_VARIANT_COMPARISON_EXPORT = False

# Choose a single selected configuration by default; original screening grids
# can initiate 144 costly training runs. Inspect with --dry-run first.
EXPERIMENT_CONFIG = "configs/selected/unet.yml"

# Dataset files must be downloaded from their publishers by the user first.
# Point to the directory containing the children: ex_dark/, lol_v1/,
# lol_v2/ and mill_s/ (NOT to its parent data/ folder).
RAW_DATA_DIRECTORY = ""           # e.g. /kaggle/input/my-raw-data/data/raw
# Alternatively, point to a directory containing both the 5 processed
# manifest CSVs + summary CSV and dataset image folders.
PREPROCESSED_DATA_DIRECTORY = ""  # e.g. /kaggle/input/my-preprocessed-data/data/preprocessed

# Checkpoints are local paths generated by a new training run or user-provided.
SINGLE_MODEL_CHECKPOINT = ""
FAILURE_CANDIDATES_PATH = "parsed_logs/unet_failure_candidates.json"
FOUR_FAMILY_CHECKPOINTS = {
    "unet": "",
    "illumination-guided-unet": "",
    "fac-ecl": "",
    "ecl-far": "",
}
LOG_PARSING_MODE = "screening"  # screening | multi_seed | final_eval | failure_candidates
LOGS_DIRECTORY = ""             # e.g. logs/unet_selected_retrain_seed_1
PARSED_OUTPUT_NAME = "reparsed.csv"


## Clone and initialize

A fresh clone is used if the project is not already available at `/kaggle/working/low-light-enhancement-cross-dataset`. The repository supplies code, configurations, manifests and experimental logs. PyTorch is pinned by `requirements.txt`; if your GPU runtime cannot support that wheel, resolve compatibility explicitly rather than silently changing the published environment.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/low-light-enhancement-cross-dataset")
VENV_DIR = Path("/kaggle/working/lle_venv")
VENV_PYTHON = VENV_DIR / "bin/python"

def run(*args):
    print("$", " ".join(str(x) for x in args), flush=True)
    subprocess.run([str(x) for x in args], cwd=REPO_DIR, check=True)

if not REPO_DIR.is_dir():
    if not GITHUB_REPOSITORY_URL.startswith("https://github.com/"):
        raise ValueError("Set GITHUB_REPOSITORY_URL to the published project repository first.")
    subprocess.run(["git", "clone", GITHUB_REPOSITORY_URL, str(REPO_DIR)], check=True)

if not VENV_PYTHON.exists():
    subprocess.run(["python", "-m", "venv", str(VENV_DIR)], check=True)
run(VENV_PYTHON, "-m", "pip", "install", "-r", "requirements.txt")
print("Repository:", REPO_DIR)


## Dataset downloads and exact folder layout

**Get the source datasets using the publishers' links (browse/download before running the notebook):**

| Dataset | Official download links | Extract under `data/raw/` as |
|---|---|---|
| **LOL-v1** | [Original project](https://daooshee.github.io/BMVC2018website/) / [Google Drive LOLdataset.zip](https://drive.google.com/file/d/157bjO1_cFuSd0HWDUuAmcHRJDVyWpOxB/view) | `lol_v1/our485/{low,high}` and `lol_v1/eval15/{low,high}` |
| **LOL-v2 Real + Synthetic** | [Authors' dataset page](https://github.com/flyywh/CVPR-2020-Semi-Low-Light) / [Google Drive LOL-v2.zip](https://drive.google.com/file/d/1dzuLCk9_gE2bFF222n3-7GVUlSVHpMYC/view) | `lol_v2/real_captured/{train,test}/{low,normal}` and `lol_v2/synthetic/{train,test}/{low,normal}` |
| **MILLs DSLR** | [Official MILL page](https://github.com/CVC-Color/MILL) / [MILLs-DSLR.zip](https://datasets.cvc.uab.cat/color/MILLs-DSLR.zip) / [Hugging Face](https://huggingface.co/datasets/davidserra9/MILL) | `mill_s/{train,validation,test}/{input,gt}` |
| **ExDark** | [Official download instructions](https://github.com/cs-chan/Exclusively-Dark-Image-Dataset/blob/master/Dataset/README.md) / [Google Drive ExDark.zip](https://drive.google.com/file/d/1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx/view) | `ex_dark/{bicycle,boat,bottle,bus,car,cat,chair,cup,dog,motorbike,people,table}` |

Use **MILLs-DSLR** (small DSLR images), not MILLf or MILLs-smartphone; follow the author's note about `Test13-9.png`. Download images through the original source rather than uploading their repository's code ZIP. Google Drive may require signing in. Follow each dataset's reuse terms; Kaggle uploads must comply with those terms.

**The RAW_DATA_DIRECTORY must contain this exact structure, with image files inside each leaf directory:**

```text
data/raw/
├── ex_dark/
│   ├── bicycle/   ├── boat/       ├── bottle/   ├── bus/
│   ├── car/       ├── cat/        ├── chair/    ├── cup/
│   ├── dog/       ├── motorbike/  ├── people/   └── table/
├── lol_v1/
│   ├── our485/{low,high}/
│   └── eval15/{low,high}/
├── lol_v2/
│   ├── real_captured/{train,test}/{low,normal}/
│   └── synthetic/{train,test}/{low,normal}/
└── mill_s/
    ├── train/{input,gt}/
    ├── validation/{input,gt}/
    └── test/{input,gt}/
```

`{a,b}` is shorthand for **two distinct directories**, not a literal directory name. Archive folders can be capitalized (`Real_captured/Train/Low`, `Bicycle/`); rename **folder levels** to the lowercase equivalents used in `configs/data/datasets.yml`. Preserve original image filenames and their case. Keep identical LOL-v1 and LOL-v2 Synthetic image pairs, LOL-v2 Real pairs like `low00001.png`/`normal00001.png`, and MILLs pairs like `scene10_a-10.png`/`scene10_a.png`.

**On Kaggle:** Create/attach an input whose root contains `data/raw/...` and set e.g. `RAW_DATA_DIRECTORY = "/kaggle/input/my-raw-data/data/raw"`. The copy step moves its contents into `/kaggle/working/low-light-enhancement-cross-dataset/data/raw/`. To build from raw data, set both `RUN_MANIFEST_GENERATION = True` and `RUN_OFFLINE_PREPROCESSING = True` before executing the data-preparation cell. The builder creates CSVs at `data/manifests/{ex_dark,lol_v1,lol_v2_real,lol_v2_synthetic,mill_s}.csv`; preprocessing creates the same five CSVs plus `dataset_summary.csv` under `data/preprocessed/`, with PNG images under `data/preprocessed/{ex_dark,lol_v1,lol_v2_real,lol_v2_synthetic,mill_s}/...`.

**If you already have fully processed data:** Prepare a Kaggle Input whose `data/preprocessed/` contains **the six CSVs AND all the image subdirectories**, then set `PREPROCESSED_DATA_DIRECTORY` to that directory. In this case keep `RUN_MANIFEST_GENERATION = False`, `RUN_OFFLINE_PREPROCESSING = False`, and you do not need raw images to train. An existing CSV without its corresponding images is insufficient; CSV image paths must resolve from the repository's working directory. The notebook does not automatically acquire the raw images from the Internet.

In [ ]:
def populate_directory(source_text, relative_target):
    if not source_text:
        return
    source = Path(source_text)
    if not source.is_dir():
        raise FileNotFoundError(f"Dataset directory not found: {source}")
    target = REPO_DIR / relative_target
    target.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target, dirs_exist_ok=True)
    print(f"Copied user-provided data from {source} to {target}")

if any((RUN_MANIFEST_GENERATION, RUN_OFFLINE_PREPROCESSING, RUN_TRAINING_EVALUATION,
        RUN_QUALITATIVE_EXPORT, RUN_VARIANT_COMPARISON_EXPORT)):
    populate_directory(RAW_DATA_DIRECTORY, "data/raw")
    populate_directory(PREPROCESSED_DATA_DIRECTORY, "data/preprocessed")

if RUN_OFFLINE_PREPROCESSING and not RUN_MANIFEST_GENERATION:
    manifest_dir = REPO_DIR / "data/manifests"
    expected = {"ex_dark.csv", "lol_v1.csv", "lol_v2_real.csv", "lol_v2_synthetic.csv", "mill_s.csv"}
    missing = sorted(name for name in expected if not (manifest_dir / name).is_file())
    if missing:
        raise FileNotFoundError(
            "Raw CSV manifests required by preprocessing are missing: " + ", ".join(missing)
            + ". Enable RUN_MANIFEST_GENERATION with a populated RAW_DATA_DIRECTORY."
        )
if RUN_MANIFEST_GENERATION:
    required_raw_roots = ["ex_dark", "lol_v1", "lol_v2/real_captured", "lol_v2/synthetic", "mill_s"]
    missing = [name for name in required_raw_roots if not (REPO_DIR / "data/raw" / name).is_dir()]
    if missing:
        raise FileNotFoundError(
            "Missing raw dataset folders in data/raw/: " + ", ".join(missing)
            + ". See the dataset download and folder layout cell above."
        )
    run(VENV_PYTHON, "scripts/build_manifests.py", "--config", "configs/data/datasets.yml")
if RUN_OFFLINE_PREPROCESSING:
    run(VENV_PYTHON, "scripts/preprocess_offline.py", "--config", "configs/data/preprocess.yml", "--no-progress-bar")


## Training and evaluation

Run the preparation cell first, or mount and copy preprocessed data (CSV manifests **and** PNG images). Before training, the notebook checks the image paths required by the selected training configuration, including test images when test evaluation is enabled. The configuration files in `configs/selected/` each define one training run; the screening configurations in `configs/experiments/` expand to multiple expensive runs. `--dry-run` only displays the number of resolved configurations. Training writes new outputs into local `logs/` and `checkpoints/`. Interpret generated results as a new experiment.

In [ ]:
if RUN_TRAINING_EVALUATION:
    run(VENV_PYTHON, "scripts/start_experiments.py", "--config", EXPERIMENT_CONFIG, "--dry-run")
    # Validate the image paths in the *selected* training manifest, and in
    # test manifests only when the YAML requests final test evaluation.
    # Run via the project virtualenv because it provides PyYAML.
    DATA_PREFLIGHT_SCRIPT = r"""
import csv
import sys
from pathlib import Path
import yaml

config = yaml.safe_load(Path(sys.argv[1]).read_text(encoding='utf-8'))
data = config['data']
needed = [(data['train_manifest'], {'train', 'validation'})]
if config.get('evaluation', {}).get('run_test', False):
    needed.extend((entry['manifest'], {'test'}) for entry in data.get('test_manifests', []))
errors = []
checked = 0
for manifest_name, splits in needed:
    manifest = Path(manifest_name)
    if not manifest.is_file():
        errors.append(f'Missing CSV manifest: {manifest}')
        continue
    with manifest.open(newline='', encoding='utf-8') as stream:
        records = [row for row in csv.DictReader(stream) if row.get('split') in splits]
    if not records:
        errors.append(f'No {sorted(splits)} records found in {manifest}')
    for row in records:
        for field in ('input_path', 'target_path'):
            value = row.get(field, '')
            if value:
                checked += 1
                if not Path(value).is_file():
                    errors.append(f'{manifest}: missing {field} image {value}')
                    if len(errors) >= 10:
                        break
        if len(errors) >= 10:
            break
    if len(errors) >= 10:
        break
if errors:
    raise FileNotFoundError(
        'Dataset preflight failed; mount and copy the matching images before training:\n'
        + '\n'.join(errors)
    )
print(f'Dataset preflight passed: {checked} image paths checked.')
"""
    run(VENV_PYTHON, "-c", DATA_PREFLIGHT_SCRIPT, EXPERIMENT_CONFIG)
    run(VENV_PYTHON, "scripts/start_experiments.py", "--config", EXPERIMENT_CONFIG, "--no-progress-bar")


## Log parsing and optional qualitative export

The repository includes the historical run JSONL files under `experiment_logs/` and complete parsed CSV/JSON analyses under `parsed_logs/`. New experiments generate logs in `logs/`. For re-parsing, set `LOGS_DIRECTORY` to the appropriate folder (for example `experiment_logs/unet_grid_seed_0_screening` or a new `logs/` subfolder) and select the matching parsing mode. Qualitative exports require an appropriate model checkpoint and locally available dataset images.

In [ ]:
if RUN_LOG_PARSING:
    if not LOGS_DIRECTORY:
        raise ValueError("Set LOGS_DIRECTORY to a real directory of JSONL logs.")
    run(VENV_PYTHON, "scripts/parse_logs.py", "--parsing-mode", LOG_PARSING_MODE,
        "--logs-dir", LOGS_DIRECTORY, "--output-name", PARSED_OUTPUT_NAME)

if RUN_QUALITATIVE_EXPORT:
    if not SINGLE_MODEL_CHECKPOINT:
        raise ValueError("Set SINGLE_MODEL_CHECKPOINT to an existing .pt file.")
    run(VENV_PYTHON, "scripts/export_qualitative_examples.py", "--checkpoint",
        SINGLE_MODEL_CHECKPOINT, "--candidates", FAILURE_CANDIDATES_PATH)

if RUN_VARIANT_COMPARISON_EXPORT:
    if not all(FOUR_FAMILY_CHECKPOINTS.values()):
        raise ValueError("Set all FOUR_FAMILY_CHECKPOINTS paths.")
    run(VENV_PYTHON, "scripts/export_variant_comparison_panel.py",
        "--unet-checkpoint", FOUR_FAMILY_CHECKPOINTS["unet"],
        "--illumination-guided-unet-checkpoint", FOUR_FAMILY_CHECKPOINTS["illumination-guided-unet"],
        "--fac-ecl-checkpoint", FOUR_FAMILY_CHECKPOINTS["fac-ecl"],
        "--ecl-far-checkpoint", FOUR_FAMILY_CHECKPOINTS["ecl-far"],
        "--seed", "42")


## Retrieve outputs

Kaggle output archives below contain only files generated by your **current execution**. Uploaded third-party images or checkpoints may have redistribution restrictions; do not publish output archives without reviewing them. Keep historical results and rerun results clearly distinguished.

In [ ]:
if any((RUN_TRAINING_EVALUATION, RUN_LOG_PARSING,
        RUN_QUALITATIVE_EXPORT, RUN_VARIANT_COMPARISON_EXPORT)):
    for folder_name in ("logs", "checkpoints", "qualitative"):
        folder = REPO_DIR / folder_name
        if folder.exists() and any(folder.rglob("*")):
            archive_path = REPO_DIR.parent / f"lle_{folder_name}"
            shutil.make_archive(str(archive_path), "zip", root_dir=REPO_DIR, base_dir=folder_name)
            print("Created:", str(archive_path) + ".zip")
